# Correlation Heatmap — Full Matrix with All Annotations
This notebook regenerates the correlation matrix with explicit verification that all 49 cells are annotated.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt

ROOT = Path('.')
OUT_DIR = ROOT / 'outputs' / 'figures' / 'thesis'
OUT_DIR.mkdir(parents=True, exist_ok=True)

import sys
print(f\"Python {sys.version}\")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"Seaborn: {sns.__version__}")
print(f"Matplotlib: {mpl.__version__}")

In [ ]:
# ━━ Variable names (canonical order matching X_tensor axis-2) ━━━━━━━━━━━━━━━━━
VARS = [
    'Albumin', 'ALP', 'ALT', 'AST', 'Bilirubin', 'BUN', 'Calcium',
    'Chloride', 'Creatinine', 'DiasABP', 'FiO2', 'GCS', 'Glucose',
    'HCO3', 'HCT', 'HR', 'K', 'Lactate', 'Mg', 'MAP', 'MechVent',
    'Na', 'NIDiasABP', 'NIMAP', 'NISysABP', 'O2Sat', 'PaCO2',
    'PaO2', 'pH', 'Platelets', 'RespRate', 'SaO2', 'SysABP',
    'Temp', 'TropI', 'Urine', 'WBC',
]

# Load tensor
print("Loading X_tensor.npy ...")
X = np.load(ROOT / 'data' / 'processed' / 'X_tensor.npy')   # (4000, 48, 37)
print(f"  shape: {X.shape}  NaN fraction: {np.isnan(X).mean():.4f}")

# Flatten to (192000, 37)
X_flat = X.reshape(-1, 37)
df = pd.DataFrame(X_flat, columns=VARS)
print(f"  flat shape: {df.shape}  rows with any value: {df.notna().any(axis=1).sum():,}")

In [ ]:
# ━━ Select key clinical variables and compute full correlation matrix ━━━━━━━━
vars_sel = ['HR', 'MAP', 'GCS', 'Lactate', 'Creatinine', 'BUN', 'Temp']
corr = df[vars_sel].corr().round(2)

print("\n=== CORRELATION MATRIX (Full 7×7, all cells) ===")
print(corr)
print(f"\nMatrix shape: {corr.shape}")
print(f"Total cells: {corr.shape[0] * corr.shape[1]} (diagonal: {corr.shape[0]}, off-diagonal: {corr.shape[0]*(corr.shape[1]-1)})")
print(f"\nKey correlations:")
print(f"  HR–Lactate:        {corr.loc['HR', 'Lactate']}")
print(f"  Creatinine–BUN:    {corr.loc['Creatinine', 'BUN']}  ← kidney function (strong positive)")
print(f"  GCS–Lactate:       {corr.loc['GCS', 'Lactate']}  ← GCS independent (near-zero)")

In [ ]:
# ━━ PLOTTING: Full symmetric matrix with ALL annotations ━━━━━━━━━━━━━━━━━━━━
mpl.rcParams.update({
    "font.family"       : "DejaVu Sans",
    "axes.edgecolor"    : "#333333",
    "axes.linewidth"    : 0.8,
    "savefig.facecolor" : "white",
})

fig, ax = plt.subplots(figsize=(10, 8), dpi=150)
fig.patch.set_facecolor("white")

# Create heatmap with NO MASK (full symmetric matrix)
sns.heatmap(
    corr,
    annot      = True,           # ← CRITICAL: show all values
    fmt        = ".2f",          # ← format as decimal (e.g., 0.69, -0.22)
    cmap       = "RdBu_r",
    center     = 0,
    vmin       = -1,
    vmax       = 1,
    square     = True,           # ← make cells square
    linewidths = 0.5,
    linecolor  = "white",
    cbar_kws   = {
        "label": "Pearson r",
        "shrink": 0.8,
    },
    ax         = ax,
    annot_kws  = {"size": 12, "weight": "bold"},  # ← larger, bold annotations
)

ax.set_title(
    "Correlation Matrix — Key Clinical Variables (Full 7×7 Symmetric)\n"
    "PhysioNet 2012 Set-A  |  n = 4,000 patients, 192,000 hourly observations",
    fontsize=13,
    fontweight="bold",
    pad=15,
)

# Rotate labels for readability
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha="right", fontsize=11)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=11)

# Add footnote explaining GCS near-zero correlation
ax.text(
    0.5, -0.18,
    "NOTE: GCS shows near-zero Pearson r with all variables, yet it is the #1 SHAP predictor.\n"
    "This is correct — low linear correlation ≠ low predictive power (GCS contributions are non-linear).",
    fontsize=10,
    color="#555555",
    ha="center",
    style="italic",
    transform=ax.transAxes,
    bbox=dict(boxstyle="round,pad=0.5", facecolor="#f0f0f0", edgecolor="#cccccc", linewidth=0.5),
)

plt.tight_layout(rect=[0, 0.12, 1, 1])
plt.show()

print("\n✓ Heatmap displayed in notebook")

In [ ]:
# ━━ VERIFICATION: Count annotations in the heatmap ━━━━━━━━━━━━━━━━━━━━━━━━━━
print("=== VERIFICATION ===")
print(f"Matrix is {corr.shape[0]} × {corr.shape[1]} = {corr.shape[0] * corr.shape[1]} total cells")
print(f"\nAll values in matrix (should be 49 total):")
print(corr.values.flatten())
print(f"\nNon-NaN cells: {np.isfinite(corr.values).sum()} / {corr.size}")
print(f"All cells have values: {np.isfinite(corr.values).all()}")

# Explicitly list key cells
print(f"\n--- Key Cell Values (should all be annotated on heatmap) ---")
print(f"HR–HR (diagonal):          {corr.loc['HR', 'HR']}")
print(f"HR–MAP (off-diagonal):     {corr.loc['HR', 'MAP']}")
print(f"MAP–HR (symmetric pair):   {corr.loc['MAP', 'HR']}  ← should equal HR–MAP")
print(f"GCS–Lactate:               {corr.loc['GCS', 'Lactate']}")
print(f"Lactate–GCS (symmetric):   {corr.loc['Lactate', 'GCS']}  ← should equal GCS–Lactate")
print(f"Creatinine–BUN:            {corr.loc['Creatinine', 'BUN']}  ← strong positive (kidney function)")
print(f"BUN–Creatinine (symmetric):{corr.loc['BUN', 'Creatinine']}  ← should equal Creatinine–BUN")

In [ ]:
# ━━ SAVE high-resolution PNG and PDF ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
png_path = OUT_DIR / "correlation_heatmap_COMPLETE.png"
pdf_path = OUT_DIR / "correlation_heatmap_COMPLETE.pdf"

fig.savefig(png_path, dpi=300, facecolor="white", bbox_inches="tight")
fig.savefig(pdf_path, dpi=300, facecolor="white", bbox_inches="tight")
plt.close(fig)

print(f"[✓] Saved: {png_path}")
print(f"[✓] Saved: {pdf_path}")